# [IAPR][iapr]: Final project - Chocolate Recognition


**Moodle group ID:** 43  
**Kaggle challenge:** `Deep learning` (either `Classic` or `Deep learning`)  
**Kaggle team name (exact):** "Choco Hunters"  

**Author 1 (sciper):** Ewa Miazga (367059)  
**Author 2 (sciper):** Sameh Lahouar (300454)   
**Author 3 (sciper):** Nour Guermazi (314474) 

**Due date:** 21.05.2025 (11:59 pm)


## Key Submission Guidelines:
- **Before submitting your notebook, <span style="color:red;">rerun</span> it from scratch!** Go to: `Kernel` > `Restart & Run All`
- **Only groups of three will be accepted**, except in exceptional circumstances.


[iapr]: https://github.com/LTS5/iapr2025

---

## 00. Imports

In [78]:
# ⚙️ Standard library
import json
import os
import shutil

# 🧪 Third-party libraries
import cv2
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split  # if needed later

# 🔥 PyTorch & Torchvision
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
from torchvision import transforms
from torch.utils.data import DataLoader, random_split
from torchvision.datasets import CocoDetection

# 🧱 Project modules
from helper import (
    collate_fn,
    draw_boxes_on_image,
    get_device,
    get_transform,
    patches_from_coco,
    save_patches
)
from loader import (
    ChocolatePatchDataset,
    CocoDataset,
    PatchTestDataset,
    UnlabeledImageFolder,
    VOCDataset
)
from models.cnn import SimpleCNN
from models.mobile import LightFasterRCNNMobileNetV3
from trainer import Trainer

# ⚙️ Device setup
device = get_device()
print(f"[INFO] Using device: {device}")

# 📁 Dataset paths
DATASET_PATH = "dataset_project_iapr2025"
VOC_DATASET_PATH = f"{DATASET_PATH}_voc"

[INFO] Using device: mps


## 01. Repo Preparation
Check if models are trained & datasets are ready

In [79]:
TRAINED_MODELS_DIR = "output"

DATASETS_DIR = "dataset_project_iapr2025"
VOC_DATASET_PATH = f"{DATASET_PATH}_voc"

In [80]:
if not os.path.exists(DATASETS_DIR):
    print(f"[WARNING] The '{DATASETS_DIR}' folder does not exist.")
    print("➡️  Please run `create_dataset.ipynb` first to generate the required datasets.")

# check if dataset_dir has the required folders
REQUIRED_FOLDERS = [
    "references",
    "test",
    "test_patches",
    "train",
    "train_annotated",
    "train_patches"
]
for folder in REQUIRED_FOLDERS:
    folder_path = os.path.join(DATASETS_DIR, folder)
    if not os.path.exists(folder_path):
        print(f"[WARNING] The '{folder_path}' folder does not exist.")
        print("➡️  Please run `create_dataset.ipynb` first to generate the required datasets.")

if not os.path.exists(VOC_DATASET_PATH):
    print(f"[WARNING] The '{VOC_DATASET_PATH}' folder does not exist.")
    print("➡️  Please download the dataset from the Kaggle website in voc format.")

if not os.path.exists(TRAINED_MODELS_DIR) or len(os.listdir(TRAINED_MODELS_DIR)) == 0:
    print(f"[WARNING] The '{TRAINED_MODELS_DIR}' folder does not exist or is empty.")
    print("➡️  Please run `train_models.ipynb` first to generate the required models.")


## 02. Data and Data Augumentation

## 03. Models

## 03.a SSDLite + MobileNetV3 Architecture

The model is based on **SSDLite**, a lightweight variant of the Single Shot MultiBox Detector (SSD), paired with a **MobileNetV3** backbone. It processes input images of size 320×320 and extracts multi-scale feature maps using depthwise separable convolutions and inverted residual blocks.

These feature maps are passed through a series of **auxiliary detection layers**, allowing the model to detect objects at different scales. The architecture is optimized for **speed and efficiency**, making it suitable for real-time inference on low-power devices while still providing reliable object detection performance.

![SSDLite Architecture](images/ssdlite_architecture.png)

*Figure: SSDLite object detection pipeline with a MobileNet backbone and 320×320 input. The model extracts multi-scale features used for bounding box generation. This diagram shows MobileNetV2, which is conceptually similar to the MobileNetV3 backbone used in our implementation.*

### Evaluation on validation set - quantitive results

![SSDLite Evaluation](images/ssd-evaluation-res.png)

### Qualitative results

|  pic 1 |  pic 2 | pic 3  |
|--------|--------|--------|
| ![](images/ssd-eval-1.png) | ![](images/ssd-eval-2.png) | ![](images/ssd-eval-3.png) |

### Prediction on test dataset

Put the score and a comment about it - maybe lets add a photo 

## 03.b SSDLite + CNN Architecture

This two-stage pipeline combines **SSDLiteMobileNetV3** for detection and a custom **SimpleCNN** for classification. The architecture reuses the pretrained backbone from the detection model to extract patches of chocolates from full images. These patches are then passed individually to a lightweight CNN to classify the type of chocolate.

|  cnn   |ssd+cnn |
|--------|--------|
| ![](images/cnn_architecture.png) | ![](images/ssd_cnn_architecture.png)

### Evaluation on validation set - quantitive results

![SSDLite Evaluation](images/cnn-evaluation-res.png)

### Prediction on test dataset

### Discussions


While the SSDLite backbone performs well in locating chocolates under controlled conditions, its performance drops in noisier scenes — for instance, when multiple unrelated objects (e.g., packaging, shadows, other shapes) are present in the image.

Since the CNN operates entirely on the patches produced by the detector, its classification accuracy is tightly linked to the quality of detections. If the detector:
	•	Misses a chocolate → the CNN never sees it
	•	Produces noisy or poorly localized crops → the CNN struggles to classify correctly

This architecture is valuable for evaluating how well a simple CNN can generalize when relying on a real-world detection front-end. It helps us test the robustness of the classifier in the presence of imperfect upstream inputs.

## 03.c Faster R-CNN + MobileNetV3 with FPN
This model was the one chosen for submission, it's executed directly in the main.py file and generates the csv file submitted on Kaggle.
The submitted versions runs only on an annotated version of the original training data and it yielded a score of 0.96951.

The architecture is based on Faster R-CNN, a two-stage object detection framework, integrated with a lightweight MobileNetV3 backbone and a custom Feature Pyramid Network (FPN). This setup leverages both high-level semantics and fine spatial details by combining multi-scale feature maps extracted from intermediate layers of the backbone.

Specifically, layers 5 and 9 of the MobileNetV3-Large model are used to generate feature maps with 40 and 80 channels respectively. These are fed into an FPN to produce high-resolution features with uniform dimensionality (256 channels) for both the Region Proposal Network (RPN) and the Region of Interest (ROI) heads.

The RPN proposes candidate object regions using an anchor generator tailored with customized sizes and aspect ratios. These proposals are then classified and refined by the ROI heads, which consist of a two-layer MLP head followed by a class-specific bounding box regressor.

The model has been carefully balanced to remain under 12 million parameters(10453044 parameters to be exact), making it computationally efficient while maintaining competitive detection accuracy.

![Number of Parameters](images/NbParams.jpg)

*Figure: Detailed number of parameters of the model*

![FCNN structure](images/FCNN_structure.png)

*Figure: The custom Faster R-CNN pipeline using a MobileNetV3 backbone with FPN. Feature maps from intermediate layers are combined and fed to the RPN and ROI heads for region proposals and object classification.*


### Evaluation on validation set - quantitive results

The dataset was split into 81 images for the training data and 9 images for the validation data. 

![Validation Predicition](images/ValPred.png)
*Figure: sample of the predictions on the validation dataset with the names of the classes written on the bounding boxes and the confidence scores*

![Qunatitavie results](images/QuantResultsFCNN.jpg)
*Figure: Overall and per class scores of the trained model on the validation dataset* 

The following graph shows the evolution of the validation loss as a function of the trained epochs, we can see that the value starts to converge around 100 epochs which may indicate that the training could stop there for similar results.
![Loss Plot](images/TrainingPlot.png)
*Figure: Plot of the Validation loss as a function of trained epochs* 


### Model Training on augmented dataset

We also managed to run our data on the augmented dataset described higher above, we chose not to submit is as our model since we can't upload our augmented data and since it only yielded a 0.97 score which is very slightly higher than the score yielded with the normal data. The fact that the increase in the performance is not much higher can be explained with the fact that we only trained it on 100 epochs(comparing to 150 for the normal dataset) due to our lack of hardware for running and the expected consequent lenght of training( around 5 hours more than for 100 epochs)

Even if we didn't choose to submit this version of our model these are the scores calculated on the validation dataset(27 images out of 270)

Here we can see that unlike the previous iteration of this model, 3 classes don't have a perfect F1 score, this is explained by the fact that the validation dataset is bigger (27 images compaing to 9) and thus its represents better the overall performance of prediction of our 
model

![Validation prediction](images/ValPredAug.png)
*Figure: sample of the predictions on the validation dataset with the names of the classes written on the bounding boxes and the confidence scores* 

![Qunatitavie results](images/QuantResultsFCNNAug.jpg)
*Figure: Overall and per class scores of the trained model on the validation dataset from the augmented dataset* 

The following graph shows the evolution of the validation loss as a function of the trained epochs, here we can see that the validation loss was decreasing and could decrease even more with more training, which can explain why we didn't get the expected performance increase.
![Loss Plot](images/TrainingPlotAug.png)
*Figure: Plot of the Validation loss as a function of trained epochs* 


## 04. Results & Discussion

|